# Regular Expressions

A **regular expression** (regex) is a compact pattern that describes a *set of strings*. Regexes are used everywhere text is processed: searching, validating input (emails, phone numbers), find-and-replace, splitting log files, and tokenizing source code.

Python's built-in [`re`](https://docs.python.org/3/library/re.html) module provides regular expressions, and we use it throughout this notebook. Later, in the **Regular Languages** notebook (14), we will see that regexes describe exactly the languages a **finite state machine** (notebook 13) can recognize.

## The Building Blocks

A regex is read left to right and is built from a few kinds of pieces:

| Piece | Meaning | Example | Matches |
|---|---|---|---|
| literal | the character itself | `cat` | the text "cat" |
| `.` | any single character | `c.t` | "cat", "cot", "c9t" |
| `[...]` | a character **class** (any one listed) | `[aeiou]` | a single vowel |
| `[a-z]` | a range | `[0-9]` | a single digit |
| `[^...]` | **negated** class | `[^0-9]` | any non-digit |
| `*` | zero or more of the previous | `ab*` | "a", "ab", "abb", … |
| `+` | one or more | `ab+` | "ab", "abb", … |
| `?` | zero or one (optional) | `colou?r` | "color", "colour" |
| `{n,m}` | between n and m repetitions | `a{2,4}` | "aa", "aaa", "aaaa" |
| `\|` | alternation (or) | `cat\|dog` | "cat" or "dog" |
| `( )` | grouping / capture | `(ab)+` | "ab", "abab", … |

There are also handy shorthand classes: `\d` (digit), `\w` (word character: letter, digit, or underscore), `\s` (whitespace), and their uppercase negations `\D`, `\W`, `\S`.

## Searching with the `re` Module

The two most common entry points are:

- `re.search(pattern, text)` — find the **first** place the pattern matches anywhere in the text.
- `re.match(pattern, text)` — match only at the **beginning** of the text.

Both return a **match object** (truthy) or `None`. Use a raw string `r"..."` for patterns so backslashes are passed through to the regex engine.

In [1]:
import re

text = "Order #4521 shipped on 2026-06-04."

m = re.search(r"\d+", text)            # first run of digits
print("search '\\d+':", m.group(), "at position", m.span())

print("match  '\\d+' at start:", re.match(r"\d+", text))      # None: text starts with 'O'
print("match  '\\w+' at start:", re.match(r"\w+", text).group())  # 'Order

search '\d+': 4521 at position (7, 11)
match  '\d+' at start: None
match  '\w+' at start: Order


## Finding All Matches, and Substituting

- `re.findall(pattern, text)` returns a list of all (non-overlapping) matches.
- `re.sub(pattern, replacement, text)` replaces every match.
- `re.compile(pattern)` pre-compiles a pattern you intend to reuse, which is faster and reads nicely.

In [2]:
import re

text = "Call 555-1234 or 555-9876, or maybe 555-0000."

print("all phone numbers:", re.findall(r"\d{3}-\d{4}", text))

# Collapse runs of whitespace down to a single space.
messy = "too    much\t\twhitespace\n\nhere"
print("normalized:", repr(re.sub(r"\s+", " ", messy)))

# Pre-compile a pattern and reuse it.
vowels = re.compile(r"[aeiou]")
print("vowels in 'sequence':", vowels.findall("sequence"))
print("devowelled:", vowels.sub("*", "sequence"))

all phone numbers: ['555-1234', '555-9876', '555-0000']
normalized: 'too much whitespace here'
vowels in 'sequence': ['e', 'u', 'e', 'e']
devowelled: s*q**nc*


## Anchors and Capture Groups

**Anchors** match positions rather than characters:

- `^` — start of the string, `$` — end of the string.
- `\b` — a word boundary (between a `\w` and a non-`\w`).

**Capture groups** `( )` let you pull pieces out of a match with `.group(n)` or `.groups()`. Anchoring a pattern with `^...$` is the usual way to check that an **entire** string matches a format.

In [3]:
import re

# Split a date into its three captured parts.
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", "Date: 2026-06-04")
print("whole match:", m.group(0))
print("groups     :", m.groups())
print("year       :", m.group(1))

# \b finds whole words only.
print("whole-word 'cat':", re.findall(r"\bcat\b", "cat category bobcat cat"))

whole match: 2026-06-04
groups     : ('2026', '06', '04')
year       : 2026
whole-word 'cat': ['cat', 'cat']


## Validating Formats

A common job is to confirm a whole string has a required shape. Anchor the pattern with `^` and `$` and test the match. Below we validate simple email addresses and US phone numbers.

In [4]:
import re

email_pattern = re.compile(r"^[\w.+-]+@[\w-]+\.[\w.-]+$")
emails = ["alice@example.com", "bob.smith@mail.co.uk", "no-at-sign.com", "x@y"]
for e in emails:
    print(f"{e:<24} valid: {bool(email_pattern.match(e))}")

print()
phone_pattern = re.compile(r"^\(\d{3}\) \d{3}-\d{4}$")
phones = ["(123) 456-7890", "123-456-7890", "(12) 345-6789"]
for p in phones:
    print(f"{p:<18} valid: {bool(phone_pattern.match(p))}")

alice@example.com        valid: True
bob.smith@mail.co.uk     valid: True
no-at-sign.com           valid: False
x@y                      valid: False

(123) 456-7890     valid: True
123-456-7890       valid: False
(12) 345-6789      valid: False


## Worked Examples

A few examples solved with Python. (The exercises in the next section are for you to solve — their solutions live in the matching notebook in `solutions/`.)

### Example 1: Extracting All Numbers

Pull every integer out of a sentence and add them up.

In [5]:
import re

text = "Room 12 has 3 chairs and 25 books across 4 shelves."
numbers = [int(n) for n in re.findall(r"\d+", text)]
print("numbers found:", numbers)
print("sum:", sum(numbers))

numbers found: [12, 3, 25, 4]
sum: 44


### Example 2: Validating a 24-Hour Time

Write a pattern that matches a valid 24-hour time `HH:MM` (00:00 to 23:59).

In [6]:
import re

pattern = re.compile(r"^([01]\d|2[0-3]):[0-5]\d$")
for t in ["09:30", "23:59", "24:00", "9:5"]:
    print(f"{t:>6}: {bool(pattern.match(t))}")

 09:30: True
 23:59: True
 24:00: False
   9:5: False


## Regular Expressions Practice Problems

Write your Python in the code cell beneath each problem (some starter code is provided). Worked solutions are in [`solutions/03_regular_expressions_key.ipynb`](solutions/03_regular_expressions_key.ipynb).

### Problem 1: Match Only Digits

Write a regular expression that matches a string consisting of **only** one or more digits, and use it to test each string in `tests`.

In [ ]:
import re
tests = ["12345", "12a45", "", "007"]

# WRITE YOUR CODE BELOW


### Problem 2: Validate a ZIP Code

Write a regex for a US ZIP code: exactly five digits, optionally followed by a hyphen and four more digits (e.g. `90210` or `90210-1234`). Test it on `tests`.

In [ ]:
import re
tests = ["90210", "90210-1234", "9021", "90210-12"]

# WRITE YOUR CODE BELOW


### Problem 3: Extract Words

Use `re.findall` to extract every word from the sentence in `text`.

In [ ]:
import re
text = "the quick brown fox jumps"

# WRITE YOUR CODE BELOW


### Problem 4: match vs. search

Show the difference between `re.match` and `re.search` by applying both with the pattern `fox` to the string `\"the fox runs\"`.

In [ ]:
import re
text = "the fox runs"

# WRITE YOUR CODE BELOW
